# 📊 RepoTopology: 4-Baseline Ablation Benchmark in Google Colab
### *Google - The Gemma 4 Developer Agent Competition*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/catlikeflyer/repotopology/blob/main/notebooks/02_repotopology_benchmark.ipynb)

This notebook evaluates **RepoTopology's 3-tiered AST context pruning** against three ablations:
- **Baseline 0**: Grep / Raw File Reading
- **Baseline 1**: Dense Vector RAG Alone (`search_similar_code`)
- **Baseline 2**: Unpruned Graph Agent (full code for all neighbors)
- **Baseline 3**: RepoTopology (Graph Traversal + 3-Tier Pruning)

**Colab Runtime Setup:**
- Standard CPU or GPU runtime is supported.
- Data can be mounted directly from Google Drive without local file transfer.

## 1. Setup & Install RepoTopology

In [ ]:
import os
import sys
from pathlib import Path

# If running inside Colab and repo not present, clone it
if not os.path.exists("repotopology") and not os.path.exists("pyproject.toml"):
    !git clone https://github.com/catlikeflyer/repotopology.git repotopology
    %cd repotopology

# Install repotopology in editable mode
!pip install -q -e "."

from repotopology.colab_utils import is_colab, mount_google_drive, link_drive_data, print_environment_summary
print_environment_summary()

## 2. Mount Dataset (Google Drive or Auto-Generated Sample)

In [ ]:
# Mount Drive if available
mount_google_drive()

# Check if Google Drive data exists; otherwise create self-contained benchmark data
DATA_DIR = Path("./data")
DRIVE_DATA = Path("/content/drive/MyDrive/repology_data")

if DRIVE_DATA.exists() and (DRIVE_DATA / "tasks.jsonl").exists():
    link_drive_data(drive_data_folder="repology_data", target_data_dir=DATA_DIR)
    print("[DATA] Using full competition dataset from Google Drive.")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if not (DATA_DIR / "tasks.jsonl").exists():
        from repotopology.evaluator import create_sample_dataset
        print("[DATA] Generating synthetic test suite for fast verification...")
        create_sample_dataset(DATA_DIR)
        print("[DATA] Synthetic dataset ready at ./data/")

## 3. Run Benchmark Evaluator (4 Baselines)

In [ ]:
import json
import pandas as pd
from repotopology.evaluator import BenchmarkEvaluator

evaluator = BenchmarkEvaluator(data_dir=DATA_DIR, top_k=5, max_steps=8)

print("Running 4-baseline comparative evaluation...")
summary = evaluator.run_all(max_tasks=5)  # Set max_tasks=None for full run

# Format results into DataFrame
rows = []
for mode_key, data in summary.get("baselines", {}).items():
    rows.append({
        "Baseline": data["label"],
        "Localization Hit Rate": f"{data['localization_hit_rate'] * 100:.1f}%",
        "Token Efficiency": f"{data['mean_token_efficiency_ratio']:.4f}",
        "Mean Prompt Tokens": f"{data['mean_prompt_tokens']:.1f}",
        "Mean Steps": f"{data['mean_trajectory_length']:.2f}",
        "Hallucination Rate": f"{data['mean_structural_hallucination_rate'] * 100:.1f}%"
    })

df = pd.DataFrame(rows)
display(df)

## 4. Export & Save Evaluation Summary

In [ ]:
out_file = Path("./eval_summary.json")
out_file.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"Evaluation results saved to {out_file}")

# Save copy to Google Drive if available
if os.path.exists("/content/drive/MyDrive"):
    drive_dest = Path("/content/drive/MyDrive/eval_summary.json")
    drive_dest.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(f"[SUCCESS] Saved backup to Google Drive at: {drive_dest}")